# Class Test CA1: Individual Assessment
## Assignment Details
**Module Title:** Data Analytics
**CRN:** 23631
**Student:** Rory Patrick Sheridan - A00082654
**Dataset:**
Annual observations for 25 existing branches of a service company over 2020-2026, recording revenue, local income, competition, customer rating, staff quality, branch type, region and monthly footfall, `ServiceLocation.csv`.
**Task:**
Answer the multiple-answer questions, develop a regression model that predicts next-year branch revenue and use it to recommend an investment site, and apply Bayes' theorem to a fraud-detection decision.

Github Repository:
https://github.com/Ri-Dearg/data-test
Github Notebook:
https://github.com/Ri-Dearg/data-test/blob/main/ca1_class_test.ipynb

### Note
- This notebook follows the brief's sections and question numbers. Each answer is in the markdown cell under its question, with the supporting code and output below it.
- Coefficients are reported to 4 decimal places, and the Question 7 predictions are calculated from those reported values.

## Table of Contents

<details>
<summary><a href="#tools--libraries">Tools & Libraries</a></summary>

- [Libraries](#libraries)
- [Utilities](#utilities)
- [Global Variables](#global-variables)

</details>

<details>
<summary><a href="#section-a-multiple-answer-questions">Section A: Multiple-Answer Questions</a></summary>

- [Question 1](#question-1)
- [Question 2](#question-2)
- [Question 3](#question-3)
- [Question 4](#question-4)
- [Question 5](#question-5)

</details>

<details>
<summary><a href="#section-b-applied-data-analysis">Section B: Applied Data Analysis</a></summary>

- [Dataset: ServiceLocation](#dataset-servicelocation)
- [Question 6: Understand and Model the Data](#question-6-understand-and-model-the-data)
- [Question 7: Make a Management Decision](#question-7-make-a-management-decision)

</details>

<details>
<summary><a href="#section-c-bayes-theorem-and-decision-making">Section C: Bayes' Theorem and Decision-Making</a></summary>

- [Question 8](#question-8)

</details>

<details>
<summary><a href="#references">References</a></summary>

- [AI Use Disclaimer](#ai-use-disclaimer)

</details>

## Tools & Libraries
1. [Matplotlib](https://matplotlib.org) - For plotting graphs.
2. [Numpy](https://numpy.org) - Calculations
3. [Pandas](https://pandas.pydata.org) - Dataframes and data manipulation
4. [Scikit-learn](https://scikit-learn.org/stable/) - For mutual information
5. [Scipy](https://scipy.org) - For the Shapiro-Wilk test, Q-Q plot and t-distribution
6. [Seaborn](https://seaborn.pydata.org) - Plot library
7. [Statsmodels](https://www.statsmodels.org/stable/index.html) - For regression models and their diagnostics
8. [Jupyter](https://jupyter.org) - Notebook Tool

### Libraries
Libraries used.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy import stats
from sklearn.feature_selection import mutual_info_regression
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.stats.outliers_influence import variance_inflation_factor

## Section A: Multiple-Answer Questions

### Question 1
A positive numerical variable has Mean = 82, Median = 54, and Skewness = +1.7. Which statements are defensible?

**Answer: A, B and D**

### Question 2
For two variables, Pearson correlation is r = 0.03, but the scatterplot shows a clear curved relationship. Which statements are correct?

**Answer: B and C**

### Question 3
A regression model gives R² = 0.91, but its residual plot shows a clear repeating pattern over time. Which statements are defensible?

**Answer: B, C and E**

### Question 4
A hypothesis test gives p = 0.018 at α = 0.05. Which statements are defensible?

**Answer: A, C and E**

### Question 5
A categorical variable Region has four categories: Dublin, Cork, Galway and Limerick. Which statements are defensible?

**Answer: A, B and E**

## Section B: Applied Data Analysis
Section B uses `ServiceLocation.csv`. Basic checks of the dataset come first. The treatment of each issue they find is justified at the start of Question 6.

### Dataset: ServiceLocation

#### Data Overview
The dataset has 175 rows: 25 branches, each observed in every year from 2020 to 2026, with 5 branches in each of the 5 regions. Annual revenue ranges from €0.07 million to €2.99 million.

In [ ]:
# Region names for the nominal Location_Code, as given in the brief
REGIONS = {1: 'Midlands', 2: 'North', 3: 'East', 4: 'South', 5: 'West'}

# Import the dataset, ordered by branch and year
df = pd.read_csv('ServiceLocation.csv').sort_values(['Branch_ID', 'Year']).reset_index(drop=True)
df

In [ ]:
# View data stats
df.describe().round(2)

In [ ]:
# Branches per region
print(df.groupby('Location_Code')['Branch_ID'].nunique().rename(index=REGIONS))

#### Types
`Branch_ID` is text and every other variable is numeric. `Location_Code` is stored as a number but is a nominal region code, and `Modern_Type` is a 0/1 indicator (Question 6 b). `Local_Income_kEUR` and `Staff_Quality` each have one missing value.

In [ ]:
# Explore structure of the data
df.info()

#### Duplicates
There are no duplicate rows and no repeated branch-year pairs.

In [ ]:
# Check for exact duplicates and for repeated branch-year pairs
print(df.duplicated().sum())
print(df.duplicated(['Branch_ID', 'Year']).sum())

#### Missing Values
Two values are missing: `Local_Income_kEUR` for B007 in 2023 and `Staff_Quality` for B016 in 2025.

In [ ]:
# Check for missing values
df.isna().sum()

In [ ]:
# Examine the rows with missing values
df[df.isna().any(axis=1)]

#### Valid Ranges
One value is outside its documented scale: B021's 2026 customer rating is 0.8, below the minimum of 1. Its ratings in the other six years are 5.5 to 5.7. Every other variable is within its valid range.

In [ ]:
out_of_range = ~df['Customer_Rating'].between(1, 10)
df.loc[out_of_range, ['Customer_Rating']]

In [ ]:
# The out-of-range rating in context: B021's ratings in every year
df[df['Branch_ID'] == 'B021'][['Year', 'Revenue_M_EUR', 'Customer_Rating']].T